# コレログラム解析ノートブック

このノートブックは **RStudio を入れなくても、RStudio っぽく時系列解析を進める** ための最小構成です。  
目的は、時系列データに対して **コレログラム（自己相関）を可視化し、周期性の手掛かりを読むこと** です。

## できること
- サンプル時系列の生成
- 時系列プロット
- コレログラム（ACF）の計算
- ラグ 24 付近に周期性がある例の確認
- 自分のCSVに差し替えて使うためのひな形


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)

# サンプル時系列
n = 240
t = np.arange(n)
y = (
    0.03 * t
    + 1.4 * np.sin(2 * np.pi * t / 24)   # 24周期
    + 0.6 * np.sin(2 * np.pi * t / 8)    # 8周期
    + np.random.normal(0, 0.5, n)
)

df = pd.DataFrame({"time": t, "value": y})
df.head()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["time"], df["value"])
plt.title("Sample Time Series")
plt.xlabel("Time")
plt.ylabel("Value")
plt.show()


## ACF を自前で計算する

R の `acf()` に相当する最小版を Python で書きます。  
こうしておくと、statsmodels が無くても動きます。


In [ ]:
def acf_values(x, nlags=60):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.dot(x, x)
    vals = [1.0]
    for lag in range(1, nlags + 1):
        vals.append(np.dot(x[:-lag], x[lag:]) / denom)
    return np.array(vals)

acf = acf_values(df["value"], nlags=60)
acf[:10]


In [ ]:
plt.figure(figsize=(10, 4))
markerline, stemlines, baseline = plt.stem(range(len(acf)), acf)
plt.setp(markerline, markersize=4)
plt.axhline(0, linewidth=1)
plt.title("Correlogram (Autocorrelation)")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.show()


## 読み方

- ラグ 0 は必ず 1
- 正の相関が大きいラグは、その間隔で値が似やすい
- 一定間隔で山が繰り返されるなら、周期性の候補がある
- この例では **ラグ 24 付近** に強い構造が出るはずです


In [ ]:
# 上位の自己相関ラグを確認
lags = np.arange(len(acf))
important = pd.DataFrame({"lag": lags[1:], "acf": acf[1:]})
important["abs_acf"] = important["acf"].abs()
important.sort_values("abs_acf", ascending=False).head(10)


## 自分のCSVに差し替えるテンプレート

CSV に `value` 列がある前提です。  
必要なら `date` 列を使って時系列インデックス化してください。


In [ ]:
# 例:
# mydf = pd.read_csv("your_timeseries.csv")
# x = mydf["value"].to_numpy()
# acf = acf_values(x, nlags=60)

# plt.figure(figsize=(10, 4))
# plt.plot(mydf["value"])
# plt.title("Your Time Series")
# plt.show()

# plt.figure(figsize=(10, 4))
# plt.stem(range(len(acf)), acf)
# plt.title("Your Correlogram")
# plt.show()


## R で書くならどうなるか

```r
x <- ts(your_vector)
acf(x, lag.max = 60, main = "Correlogram")
```

つまり今回のノートブックは、**RStudio の代わりにブラウザや Colab で動かせる軽量版** です。
